# Finding Refusal Probes

In [1]:
import shlex
import subprocess
import sys

from pathlib import Path

from probelab.experiments.refusal.pipeline import CONFIGS

EXPECTED_CONFIGS = {c.name for c in CONFIGS}
print("Configs the pipeline runs per model:", sorted(EXPECTED_CONFIGS))

Configs the pipeline runs per model: ['arditi_last_token', 'last_5_mean', 'post_instruction_mean']


In [ ]:
REPO_ROOT   = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RESULTS_DIR = REPO_ROOT / "results" / "refusal"

MODELS = [
    "meta-llama/Meta-Llama-3.1-8B-Instruct",
    "meta-llama/CodeLlama-13b-Instruct-hf",
    "google/gemma-4-26B-A4B-it",
#    "Qwen/Qwen3.6-27B-FP8",
]

def slug(model_id: str) -> str:
    return model_id.replace("/", "__")

def out_dir(model_id: str) -> Path:
    return RESULTS_DIR / slug(model_id)

In [3]:
def run_model(model_id: str, force: bool = False) -> None:
    od = out_dir(model_id)
    od.mkdir(parents=True, exist_ok=True)

    completed = {
        p.name for p in od.iterdir()
        if p.is_dir() and (p / "causal.json").exists()
    }

    if not force and EXPECTED_CONFIGS.issubset(completed):
        print(f"[orchestrator] {model_id}: all configs complete, skipping (force=True to redo)")
        return

    if completed and not force:
        missing = sorted(EXPECTED_CONFIGS - completed)
        print(f"[orchestrator] {model_id}: resuming — missing {missing}")

    cmd = [sys.executable, "-m", "probelab.experiments.refusal.pipeline",
           model_id, "--out", str(od)]
    log_path = od / "run.log"
    
    print(f"[orchestrator] {model_id}: launching (log -> {log_path})\n  {shlex.join(cmd)}\n", flush=True)

    with log_path.open("w") as logf:
        proc = subprocess.Popen(
            cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, bufsize=1,
        )

        for line in proc.stdout:
            logf.write(line); logf.flush(); print(line, end="")

        rc = proc.wait()

    if rc != 0:
        raise RuntimeError(f"{model_id} pipeline failed (exit code {rc}). Log: {log_path}")

    print(f"\n[orchestrator] {model_id}: done\n")

# Make sure ANTHROPIC_API_KEY is in the subprocess env. Set it once in your
# shell before launching jupyter, or uncomment the next line.
# import os; os.environ["ANTHROPIC_API_KEY"] = "sk-ant-..."

for mid in MODELS:
    run_model(mid)


[orchestrator] meta-llama/Meta-Llama-3.1-8B-Instruct: all configs complete, skipping (force=True to redo)
[orchestrator] meta-llama/CodeLlama-13b-Instruct-hf: all configs complete, skipping (force=True to redo)
[orchestrator] google/gemma-4-26B-A4B-it: all configs complete, skipping (force=True to redo)


In [4]:
from probelab.experiments.refusal import load_runs

runs = load_runs(RESULTS_DIR, MODELS)

for mid, r in runs.items():
    print(f"\n{mid}  ({r.n_layers} transformer layers)")

    for cname, c in r.configs.items():
        print(f"  {cname:25s}  best probe layer {c.best_probe_layer:>3}  "
              f"(dev={c.dev_accs[c.best_probe_layer]:.2%})  ||  "
              f"best causal layer {c.best_causal_layer:>3}  "
              f"(refusal {c.causal_baseline:.2%} -> "
              f"{c.causal_per_layer[c.best_causal_layer]:.2%})")


meta-llama/Meta-Llama-3.1-8B-Instruct  (32 transformer layers)
  arditi_last_token          best probe layer  11  (dev=99.46%)  ||  best causal layer  11  (refusal 90.22% -> 0.00%)
  last_5_mean                best probe layer   7  (dev=99.46%)  ||  best causal layer  13  (refusal 90.22% -> 0.00%)
  post_instruction_mean      best probe layer   7  (dev=99.46%)  ||  best causal layer  12  (refusal 90.22% -> 1.25%)

meta-llama/CodeLlama-13b-Instruct-hf  (40 transformer layers)
  arditi_last_token          best probe layer  11  (dev=100.00%)  ||  best causal layer  14  (refusal 96.74% -> 5.00%)
  last_5_mean                best probe layer  10  (dev=100.00%)  ||  best causal layer  13  (refusal 96.74% -> 5.00%)
  post_instruction_mean      best probe layer  12  (dev=100.00%)  ||  best causal layer  18  (refusal 96.74% -> 1.25%)

google/gemma-4-26B-A4B-it  (30 transformer layers)
  arditi_last_token          best probe layer  17  (dev=100.00%)  ||  best causal layer  11  (refusal 80.43% -

In [5]:
from probelab.experiments.refusal import plot_refusal_by_config

for mid, r in runs.items():
    plot_refusal_by_config(r).show()

In [6]:
from probelab.experiments.refusal import plot_config_probe_similarity

for mid, r in runs.items():
    # Defaults to the best causal layer of the first config; pass layer=N to override.
    plot_config_probe_similarity(r).show()

In [7]:
from probelab.experiments.refusal import plot_refusal_across_models

for cfg_name in sorted(EXPECTED_CONFIGS):
    plot_refusal_across_models(runs, cfg_name, depth_normalize=True).show()

In [8]:
from probelab.experiments.refusal import plot_best_config_per_model

plot_best_config_per_model(runs).show()